# 07 — Movie Success Classification

For this notebook, **success means high audience score**, not commercial success.

Definition:

- `success = 1` when score >= 7.0
- `success = 0` otherwise

This definition avoids mixing financial variables with a target that is directly derived from financial variables.

We intentionally exclude `score` from the model features because score defines the target.
We also exclude gross, profit, ROI and votes to avoid leakage/post-release information.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    roc_auc_score
)

movies = pd.read_csv("../data/processed/movies_cleaned.csv")
movies = movies.dropna(how="all").reset_index(drop=True)

for col in ["year", "score", "runtime", "release_year"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

for col in ["genre", "rating", "country", "release_month"]:
    movies[col] = movies[col].fillna("Unknown")

movies["success"] = (movies["score"] >= 7.0).astype(int)

print(movies["success"].value_counts())
print(movies["success"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
features = [
    "year", "runtime", "release_year",
    "genre", "rating", "country", "release_month"
]

data = movies[features + ["success"]].dropna().copy()

X = data[features]
y = data["success"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_features = ["year", "runtime", "release_year"]
categorical_features = ["genre", "rating", "country", "release_month"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

In [ ]:
baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 4))

In [ ]:
success_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

success_model.fit(X_train, y_train)

pred = success_model.predict(X_test)
proba = success_model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred, zero_division=0),
    "Recall": recall_score(y_test, pred, zero_division=0),
    "F1": f1_score(y_test, pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, proba)
}

pd.Series(metrics)

In [ ]:
print(classification_report(
    y_test,
    pred,
    target_names=["Not high-rated", "High-rated"],
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not high-rated", "High-rated"],
    yticklabels=["Not high-rated", "High-rated"]
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(success_model, "../models/success_classifier.pkl")
print("Saved: ../models/success_classifier.pkl")

## Interpretation

Because success is defined using the score itself, this is a classification experiment around a clearly defined target. Do not describe it as predicting commercial success.

For a future version, commercial success could be defined using profit or ROI, but then the feature set must be designed carefully to avoid target leakage.